# 技能：自动加载代码仓库的 `.claude/skills`

已经在为 Claude Code 编写技能的团队，通常把这些技能放在代码仓库里的 `.claude/skills/<skill-name>/SKILL.md`，与它们所描述的代码一起做版本管理。如果把它们逐个上传到 Skills API，并在每个智能体上固定版本，就等于重复劳动，而且一旦有人修改仓库就会出现漂移。

当会话挂载一个 GitHub 仓库时，运行框架会在会话开始时扫描仓库根目录的 `.claude/skills/` 目录，并把发现的每个技能注入智能体的系统提示词：包括技能名称、描述，以及它在沙箱内的路径。当某个请求与技能描述相匹配时，模型会立刻用内置的 `read` 工具读取该技能的 `SKILL.md`，然后照做，包括技能自带的任何脚本或参考文件。无需上传，无需在智能体上设置 `skills` 字段，也无需版本台账。你得到的技能，就是你检出的那个分支上的技能。

本笔记本将讲解：

- 扫描器期望的磁盘布局
- 挂载仓库以便发现其中的技能
- 确认发现结果，并观察“先读取、再遵循”的协议
- 边界在哪里：布局规则、技能数量上限以及会话中途的变更

## 1. 设置客户端

你的环境中需要有 `GITHUB_TOKEN`。一个具有公开仓库读取权限的细粒度个人访问密钥（personal access token）就够了，因为本笔记本挂载的是公开仓库。仓库技能只在 Anthropic 托管环境（`anthropic_cloud`）中才会被发现。自托管沙箱不会克隆 GitHub 仓库，因此该功能在那里不在讨论范围内。

In [ ]:
%%capture
%pip install -qU "anthropic>=0.121.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
GH_TOKEN = os.environ.get("GITHUB_TOKEN")
if not GH_TOKEN:
    raise SystemExit(
        "Set GITHUB_TOKEN in your environment (a fine-grained PAT with public-repo read)."
    )
client = anthropic.Anthropic()

## 2. 扫描器寻找的布局

技能只从一个地方被发现：仓库根目录的 `.claude/skills/`，每个技能一个目录，且每个目录顶层有一个 `SKILL.md`。这与 Claude Code 使用的约定相同。

```
your-repo/
└── .claude/
    └── skills/
        ├── cookbook-audit/
        │   ├── SKILL.md          <- name + description frontmatter, then instructions
        │   ├── style_guide.md    <- reference file the instructions point at
        │   └── validate_notebook.py  <- script the instructions run
        └── another-skill/
            └── SKILL.md
```

`SKILL.md` 以 YAML frontmatter 开头，其中带有 `name` 和 `description`。描述是模型判断是否打开该文件时所匹配的内容，因此它应当读起来像一个触发条件：这个技能做什么、什么时候该用它。

```markdown
---
name: cookbook-audit
description: Audit an Anthropic Cookbook notebook based on a rubric. Use whenever a notebook review or audit is requested.
---

# Cookbook Audit
...
```

公开的 `anthropics/claude-cookbooks` 仓库正好带有这个技能，其中 `style_guide.md` 是一份评分细则，还有一个 `validate_notebook.py` 脚本。挂载该仓库，你一次性就得到了带说明、参考文件和可运行脚本的技能。

## 3. 创建一个不附带任何技能的智能体

智能体的定义中没有任何内容与技能相关。它得到的是标准工具集和一个通用提示词。技能是随仓库而来的，而不是随智能体定义而来的。

In [ ]:
env = client.beta.environments.create(
    name="repo-skills-demo",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

reviewer = client.beta.agents.create(
    name="cookbook_reviewer",
    description="Reviews notebooks in a mounted cookbook repository.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-use-skills-from-a-repo"},
    model={"id": MODEL},
    system="You review notebooks in the repository mounted under /workspace. "
    "When the repository provides a process for a task, follow it.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print(f"{reviewer.name}: {reviewer.id} v{reviewer.version}")

## 4. 挂载仓库

`github_repository` 资源会在会话开始时把仓库克隆进容器。`url` 使用普通的 `https://github.com/{owner}/{repo}` 形式（不带 `.git` 后缀，也不用 SSH 形式）。`checkout` 用于固定分支或提交，`mount_path` 会覆盖默认的 `/workspace/<repo-name>`。技能发现遵循所检出的引用，因此带有不同技能的分支会得到不同的技能。

In [ ]:
session = client.beta.sessions.create(
    agent=reviewer.id,
    environment_id=env.id,
    title="Cookbook audit",
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/anthropics/claude-cookbooks",
            "authorization_token": GH_TOKEN,
            "checkout": {"type": "branch", "name": "main"},
            # mount_path defaults to /workspace/claude-cookbooks
        }
    ],
    betas=BETAS,
)
print(session.id, session.status)

## 5. 询问智能体它有哪些技能

发现过程在会话开始时就已经完成，所以这个问题无需探索文件即可回答。智能体直接根据系统提示词说出 `cookbook-audit` 以及它的 `SKILL.md` 的路径。

In [ ]:
from utilities import stream_until_end_turn

client.beta.sessions.events.send(
    session.id,
    events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "What skills are available to you in this repo?"}],
        }
    ],
    betas=BETAS,
)
stream_until_end_turn(client, session.id)

## 6. 不点名也能触发技能

描述的意义在于让模型按意图路由。这个请求从未提到“技能”或“cookbook-audit”，它要的是这个技能所能产出的结果。

注意工具事件。本轮中第一次 `read` 打开的是该技能的 `SKILL.md`。随后智能体按照技能的指示行事：读取 `style_guide.md`、运行 `validate_notebook.py`，并对照评分细则打分，而不是自己发明一套审计工作流。

In [ ]:
client.beta.sessions.events.send(
    session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Do a notebook audit of "
                    "managed_agents/CMA_iterate_fix_failing_tests.ipynb "
                    "against the cookbook style guide, and give me the score.",
                }
            ],
        }
    ],
    betas=BETAS,
)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            target = ev.input.get("file_path") or ev.input.get("command") or ""
            print(f"[{ev.name}] {str(target)[:100]}")
        elif ev.type == "agent.message":
            print("".join(b.text for b in ev.content if b.type == "text"), end="")
        elif ev.type == "session.status_idle":
            print(f"\n[idle] stop_reason={ev.stop_reason.type}")
            break

把 `read` 调用单独提取出来，就能单独看到这个协议：先是技能的 `SKILL.md`，然后是技能自带的参考文件和脚本，最后是这项工作涉及的任意仓库文件。

In [ ]:
reads = [
    ev.input.get("file_path")
    for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS)
    if ev.type == "agent.tool_use" and ev.name == "read"
]
for path in reads:
    print(path)

## 7. 扫描器强制执行的规则

**布局要求严格。** 只有仓库根目录的 `.claude/skills/<name>/SKILL.md` 会在会话开始时被宣告。没有子目录的裸 `.claude/skills/SKILL.md`、两层嵌套的 `.claude/skills/a/b/SKILL.md`，以及缺少 `.claude` 父目录的 `skills/<name>/SKILL.md`，都会被忽略。

**frontmatter 要求宽松。** 没有 frontmatter 的 `SKILL.md` 仍会被宣告，其名称取目录名、描述为空。过长的描述会被压平为一行，并在超过 2,000 个字符处截断。CRLF 行尾也能正常解析。

**存在数量上限。** 只有前 64 个技能目录会被宣告。超出部分的目录会被静默丢弃，因此技能库庞大的仓库应当把会话依赖的那些技能放在靠前位置并保持精简。

**扫描只运行一次。** 发现发生在会话开始时，并为该会话持久化。会话中途向仓库推送新技能，对正在运行的会话毫无影响。针对该仓库创建的下一个会话才会看到它。

**嵌套技能是惰性发现的。** 如果仓库里有 `packages/foo/.claude/skills/lint/SKILL.md` 而没有根级技能，那么会话开始时不会宣告任何内容。一旦智能体用内置的 `read` 工具读取了 `packages/foo/` 下的某个文件，运行框架就会注意到嵌套的 `.claude/skills/` 目录，并在该会话剩余时间里把 `lint` 呈现出来。通过 `bash`（`cat`）读取同一个文件不会触发它。只有 `read` 工具会。

## 8. 仓库技能与 Skills API 并存

两种机制可以在同一个会话中共存，而且两类技能都会出现在智能体的技能列表里。

当技能属于代码本身时，使用仓库技能：构建与测试约定、评审清单、部署运维手册，以及任何你希望通过拉取请求与源码一起修改的内容。无论是人还是智能体，只要克隆了仓库，得到的都是同一份说明。

当技能是跨越多个仓库的组织级资产、需要独立于任何代码库的版本管理，或者必须在完全不挂载仓库的会话中可用时，使用 [Skills API](https://platform.claude.com/docs/en/managed-agents/skills)（在智能体上设置 `skills=[{"type": "custom", "skill_id": ...}]`）。

两者之间的名称冲突是允许的：仓库里的 `verify` 技能和组织级的 `verify` 技能都会出现，路径不同，模型根据请求进行选择。如果两个被挂载的仓库都带有同名技能，两者都会带着各自不同的挂载路径被宣告。

## 9. 清理

In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")